In [ ]:
!nvidia-smi

Fri Jan 10 11:04:56 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   45C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
# This get the RAPIDS-Colab install files and test check your GPU.  Run this and the next cell only.
# Please read the output of this cell.  If your Colab Instance is not RAPIDS compatible, it will warn you and give you remediation steps.
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 566, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 566 (delta 188), reused 144 (delta 100), pack-reused 269 (from 1)
Receiving objects: 100% (566/566), 182.35 KiB | 893.00 KiB/s, done.
Resolving deltas: 100% (290/290), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
Installing RAPIDS remaining 24.10.* libraries
Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.7/567.7 MB 55.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 GB 27.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 166.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 kB 175.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 91.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import cudf
cudf.__version__

'25.02.01'

In [2]:
import cuml
cuml.__version__

'25.02.01'

In [4]:
# import cugraph
# cugraph.__version__

In [5]:
# import cuspatial
# cuspatial.__version__

In [6]:
# import cuxfilter
# cuxfilter.__version__

In [7]:
import pandas as pd

df=pd.read_csv('https://archive.ics.uci.edu/static/public/468/data.csv')
df.drop('Month',axis=1,inplace=True)
df['Weekend']=df['Weekend'].astype('int')
df['Revenue']=df['Revenue'].astype('int')
df=pd.get_dummies(df,dtype=float)

# Train and test split
# Test - Holdout set
# GridSearchCV

from sklearn.model_selection import train_test_split, GridSearchCV

X=df.drop('Revenue',axis=1)
y=df['Revenue']


#Stratified random sampling to create train and test split
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=123,stratify=y)


#Holtout sets are x_test,y_test

#What is the dimension of the feature space in our problem?
x_train.shape[1]

18

In [13]:
%%time
print("Scikit learn implementation")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score,confusion_matrix

clf=RandomForestClassifier()
model=GridSearchCV(clf,param_grid={'n_estimators':[100,200,300],'max_depth':[3,5,7],'max_features':[3,5,7]},n_jobs=-1,cv=3)
model.fit(x_train,y_train)

# model.predict(x_test)

#Average Number of correct predictions
print(model.score(x_test,y_test))

print(recall_score(y_test,model.predict(x_test)))

print(confusion_matrix(y_test,model.predict(x_test)))

Scikit learn implementation
0.9013947453778787
0.5576519916142557
[[2513   93]
 [ 211  266]]
CPU times: user 1.78 s, sys: 129 ms, total: 1.9 s
Wall time: 1min 9s


In [15]:
%%time
print("Rapids CuML implementation")
from cuml.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score,confusion_matrix

clf=RandomForestClassifier()
model=GridSearchCV(clf,param_grid={'n_estimators':[100,200,300],'max_depth':[3,5,7],'max_features':[3,5,7]},n_jobs=-1,cv=3)
model.fit(x_train,y_train)

# model.predict(x_test)

#Average Number of correct predictions
print(model.score(x_test,y_test))

print(recall_score(y_test,model.predict(x_test)))

print(confusion_matrix(y_test,model.predict(x_test)))

Rapids CuML implementation
0.899448573589325
0.5534591194968553
[[2509   97]
 [ 213  264]]
CPU times: user 703 ms, sys: 180 ms, total: 883 ms
Wall time: 30.4 s
